# Source Contribution Analysis

Which source contributed the most high-priority candidates for a given system?

This notebook loads a pipeline report JSON, groups candidates by their ingestion
source, and visualizes the contribution breakdown. Adjust the configuration cell
below to point at a different system or report.

In [ ]:
# ============================================================
# CONFIGURATION — edit these variables before running
# ============================================================
from pathlib import Path
from materials_discovery.common.io import workspace_root

SYSTEM_NAME = "Al-Cu-Fe"  # change to the system you want to analyse

# Default: look for the report under data/reports/
WORKSPACE = workspace_root()
REPORT_PATH = WORKSPACE / "data" / "reports" / f"{SYSTEM_NAME.lower().replace('-', '_')}_report.json"

# Priority thresholds (lower hifi_score = better)
HIGH_THRESHOLD = 0.3
MEDIUM_THRESHOLD = 0.5

print(f"System:      {SYSTEM_NAME}")
print(f"Report path: {REPORT_PATH}")
print(f"Exists:      {REPORT_PATH.exists()}")

In [ ]:
# ============================================================
# Load report and extract entries
# ============================================================
import json
from collections import defaultdict

if not REPORT_PATH.exists():
    print(f"[INFO] Report not found at {REPORT_PATH}")
    print("       Run the pipeline first: mdisc report --config configs/systems/al_cu_fe.yaml")
    entries = []
else:
    report_data = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    entries = report_data.get("entries", [])
    print(f"Loaded {len(entries)} report entries")
    if entries:
        print(f"Sample keys: {list(entries[0].keys())[:8]}")

In [ ]:
# ============================================================
# Group entries by source
# ============================================================
def extract_source(entry: dict) -> str:
    """Try various locations where the source key may be stored."""
    # Phase 4+ format: benchmark_context.source_keys
    ctx = entry.get("benchmark_context", {})
    if ctx:
        keys = ctx.get("source_keys", [])
        if keys:
            return ",".join(keys)
    # Phase 3 format: evidence.calibration_provenance
    evidence = entry.get("evidence", {})
    prov = evidence.get("calibration_provenance", {})
    if prov:
        return prov.get("source_key", "unknown")
    # Fallback: check top-level source
    return entry.get("source", "unknown")


def classify_priority(entry: dict) -> str:
    score = entry.get("hifi_score")
    if score is None:
        return "unknown"
    if score <= HIGH_THRESHOLD:
        return "high"
    if score <= MEDIUM_THRESHOLD:
        return "medium"
    return "watch"


by_source: dict[str, dict[str, int]] = defaultdict(lambda: {"high": 0, "medium": 0, "watch": 0, "unknown": 0})

for entry in entries:
    source = extract_source(entry)
    priority = classify_priority(entry)
    by_source[source][priority] += 1

if by_source:
    print(f"Sources found: {list(by_source.keys())}")
    for src, counts in by_source.items():
        total = sum(counts.values())
        print(f"  {src}: {total} total  (high={counts['high']}, medium={counts['medium']}, watch={counts['watch']})")
else:
    print("No entries to group — run the pipeline first.")

In [ ]:
# ============================================================
# Bar chart: candidates by source and priority
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if by_source:
    sources = sorted(by_source.keys())
    priorities = ["high", "medium", "watch"]
    colors = ["#2ecc71", "#f39c12", "#e74c3c"]

    x = np.arange(len(sources))
    width = 0.25

    fig, ax = plt.subplots(figsize=(max(6, len(sources) * 2), 5))
    for i, (priority, color) in enumerate(zip(priorities, colors)):
        values = [by_source[src][priority] for src in sources]
        ax.bar(x + i * width, values, width, label=priority.capitalize(), color=color)

    ax.set_xlabel("Source")
    ax.set_ylabel("Candidate count")
    ax.set_title(f"Source Contribution — {SYSTEM_NAME}")
    ax.set_xticks(x + width)
    ax.set_xticklabels(sources, rotation=30, ha="right")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
# ============================================================
# Table: top candidates per source (hifi_score, xrd_confidence)
# ============================================================
TOP_N = 5

if entries:
    # Sort all entries by hifi_score ascending (lower = better)
    scored = [(e.get("hifi_score", 9999), e) for e in entries]
    scored.sort(key=lambda t: t[0])

    seen_sources: set[str] = set()
    rows = []
    for _, entry in scored:
        source = extract_source(entry)
        rows.append({
            "candidate_id": entry.get("candidate_id", "n/a"),
            "source": source,
            "priority": classify_priority(entry),
            "hifi_score": entry.get("hifi_score"),
            "xrd_confidence": entry.get("xrd_confidence"),
        })

    print(f"Top {TOP_N} candidates per metric (all sources):")
    header = f"{'candidate_id':<36} {'source':<20} {'priority':<8} {'hifi_score':>10} {'xrd_conf':>10}"
    print(header)
    print("-" * len(header))
    for row in rows[:TOP_N]:
        hifi = f"{row['hifi_score']:.4f}" if row['hifi_score'] is not None else "N/A"
        xrd = f"{row['xrd_confidence']:.4f}" if row['xrd_confidence'] is not None else "N/A"
        print(f"{row['candidate_id']:<36} {row['source']:<20} {row['priority']:<8} {hifi:>10} {xrd:>10}")
else:
    print("No entries available.")

In [ ]:
# ============================================================
# Summary
# ============================================================
total = len(entries)
high_count = sum(by_source[s]["high"] for s in by_source)
medium_count = sum(by_source[s]["medium"] for s in by_source)

print(f"=== Source Contribution Summary: {SYSTEM_NAME} ===")
print(f"Total candidates analysed : {total}")
print(f"High-priority (<={HIGH_THRESHOLD} score) : {high_count}")
print(f"Medium-priority (<={MEDIUM_THRESHOLD} score): {medium_count}")
if by_source:
    best_source = max(by_source, key=lambda s: by_source[s]["high"])
    print(f"Best source (high priority): {best_source} ({by_source[best_source]['high']} candidates)")
else:
    print("Run the pipeline to generate candidates.")